# VolatilityQuant — Feature 3: Options Hedging Simulator

Demonstrates Black–Scholes pricing, dynamic delta hedging, model-based volatility inputs, Monte Carlo experiments, and interactive controls. Results are educational simulations, not live hedging backtests.

In [ ]:
!pip install -q numpy pandas scipy matplotlib yfinance arch xgboost ipywidgets

## 1. Black–Scholes pricing and option delta

In [ ]:

import numpy as np
import pandas as pd
from scipy.stats import norm

# Black-Scholes option pricing and delta
def black_scholes(S, K, T, r, sigma, option_type="call"):

    if S <= 0 or K <= 0:
        raise ValueError("Stock and strike prices must be positive.")

    if T <= 0:
        if option_type == "call":
            return max(S - K, 0), float(S > K)
        elif option_type == "put":
            return max(K - S, 0), -float(S < K)
        else:
            raise ValueError("Option type must be call or put.")

    if sigma <= 0:
        raise ValueError("Volatility must be positive.")

    d1 = (
        np.log(S / K)
        + (r + 0.5 * sigma**2) * T
    ) / (sigma * np.sqrt(T))

    d2 = d1 - sigma * np.sqrt(T)

    if option_type == "call":

        price = (
            S * norm.cdf(d1)
            - K * np.exp(-r * T) * norm.cdf(d2)
        )

        delta = norm.cdf(d1)

    elif option_type == "put":

        price = (
            K * np.exp(-r * T) * norm.cdf(-d2)
            - S * norm.cdf(-d1)
        )

        delta = norm.cdf(d1) - 1

    else:
        raise ValueError("Option type must be call or put.")

    return price, delta


# Example: hypothetical stock and option
S = 100
K = 100
T = 30 / 365
r = 0.04
sigma = 0.30

call_price, call_delta = black_scholes(
    S, K, T, r, sigma, "call"
)

put_price, put_delta = black_scholes(
    S, K, T, r, sigma, "put"
)

print("BLACK-SCHOLES OPTIONS PRICING")
print("--------------------------------")
print(f"Stock price: ${S:.2f}")
print(f"Strike price: ${K:.2f}")
print(f"Volatility: {sigma * 100:.1f}%")

print(f"\nCall price: ${call_price:.2f}")
print(f"Call delta: {call_delta:.4f}")

print(f"\nPut price: ${put_price:.2f}")
print(f"Put delta: {put_delta:.4f}")


## 2. Call and put payoffs at expiration

In [ ]:

import numpy as np
import matplotlib.pyplot as plt

# Possible stock prices at expiration
stock_prices = np.linspace(60, 140, 200)

# Option payoffs at expiration
call_payoff = np.maximum(stock_prices - K, 0)
put_payoff = np.maximum(K - stock_prices, 0)

# Profit = payoff - initial option premium
call_profit = call_payoff - call_price
put_profit = put_payoff - put_price

# Plot both options
plt.figure(figsize=(11, 6))

plt.plot(
    stock_prices,
    call_profit,
    label="Long Call",
    linewidth=2
)

plt.plot(
    stock_prices,
    put_profit,
    label="Long Put",
    linewidth=2
)

plt.axhline(0, color="black", linestyle="--")
plt.axvline(K, color="gray", linestyle=":")

plt.title("Call vs Put — Profit at Expiration")
plt.xlabel("Stock Price at Expiration ($)")
plt.ylabel("Profit per Share ($)")
plt.legend()
plt.grid(alpha=0.3)
plt.show()


## 3. Reusable self-financing delta-hedging simulator

This supersedes earlier exploratory hedge implementations. The initial option premium is common across strategies; stock trades, financing, transaction costs, and settlement are included. Assumes European-style options, no dividends, and fractional shares.

In [ ]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def simulate_hedge_corrected(
    stock_path,
    K=100,
    T=30/365,
    r=0.04,
    hedge_vol=0.30,
    market_vol=0.30,
    option_type="call",
    contract_size=100,
    transaction_cost=0.001
):
    """
    Delta hedge one long European option.

    market_vol determines the common purchase premium.
    hedge_vol determines the hedge ratios.

    Includes cash interest, stock trading costs,
    and option settlement at expiration.
    """

    stock_path = np.asarray(stock_path, dtype=float)

    if len(stock_path) < 2 or np.any(stock_path <= 0):
        raise ValueError("Invalid stock price path.")

    days = len(stock_path) - 1
    dt = T / days

    # Same market premium for every strategy
    market_price, _ = black_scholes(
        stock_path[0], K, T, r,
        market_vol, option_type
    )

    # Initial hedge ratio
    _, initial_delta = black_scholes(
        stock_path[0], K, T, r,
        hedge_vol, option_type
    )

    # Purchase one option contract
    cash = -market_price * contract_size

    # Establish stock hedge
    shares = -initial_delta * contract_size

    # Pay for or receive proceeds from stock trade
    cash -= shares * stock_path[0]

    # Deduct initial stock transaction costs
    total_costs = (
        abs(shares)
        * stock_path[0]
        * transaction_cost
    )

    cash -= total_costs

    records = []

    for day, S in enumerate(stock_path):

        remaining_T = max(T - day * dt, 0)

        # Mark option using common market volatility
        option_price, _ = black_scholes(
            S, K, remaining_T, r,
            market_vol, option_type
        )

        # Hedge delta using selected volatility estimate
        _, hedge_delta = black_scholes(
            S, K, remaining_T, r,
            hedge_vol, option_type
        )

        if day > 0:

            # Interest on cash balance
            cash *= np.exp(r * dt)

            # Rebalance before expiration
            if day < days:

                target_shares = (
                    -hedge_delta * contract_size
                )

                trade = target_shares - shares

                trade_cost = (
                    abs(trade)
                    * S
                    * transaction_cost
                )

                cash -= trade * S
                cash -= trade_cost

                total_costs += trade_cost
                shares = target_shares

        # At expiration, settle option and close stock hedge
        if day == days:

            if option_type == "call":
                payoff = max(S - K, 0)
            else:
                payoff = max(K - S, 0)

            cash += payoff * contract_size

            # Close remaining stock position
            closing_cost = (
                abs(shares)
                * S
                * transaction_cost
            )

            cash += shares * S
            cash -= closing_cost

            total_costs += closing_cost
            shares = 0

            portfolio_value = cash

        else:

            portfolio_value = (
                option_price * contract_size
                + shares * S
                + cash
            )

        records.append({
            "Day": day,
            "Stock": S,
            "HedgeDelta": hedge_delta,
            "Shares": shares,
            "PortfolioValue": portfolio_value,
            "TransactionCosts": total_costs
        })

    return pd.DataFrame(records)


# Test using our existing simulated stock path
corrected_hedge = simulate_hedge_corrected(
    stock_path,
    K=100,
    T=30/365,
    r=0.04,
    hedge_vol=0.30,
    market_vol=0.30,
    option_type="call"
)

print(
    "Final hedged P&L: $",
    round(corrected_hedge["PortfolioValue"].iloc[-1], 2)
)

print(
    "Total transaction costs: $",
    round(corrected_hedge["TransactionCosts"].iloc[-1], 2)
)

plt.figure(figsize=(11, 5))

plt.plot(
    corrected_hedge["Day"],
    corrected_hedge["PortfolioValue"],
    label="Corrected Delta Hedge",
    linewidth=2
)

plt.axhline(0, color="black", linestyle="--")

plt.title("Corrected Dynamic Delta-Hedging Simulation")
plt.xlabel("Trading Day")
plt.ylabel("Portfolio P&L ($)")
plt.legend()
plt.grid(alpha=0.3)
plt.show()


## 4. Generate volatility inputs from historical market data

Downloads NVIDIA data and fits historical, GARCH(1,1), and log-variance XGBoost estimates. Forecasts are annualized five-day estimates.

In [ ]:

import yfinance as yf
import numpy as np
import pandas as pd

from arch import arch_model
from xgboost import XGBRegressor

# Select a stock
ticker = "NVDA"
HORIZON = 5
EPSILON = 1e-6

# Download historical prices
data = yf.download(
    ticker,
    period="5y",
    auto_adjust=True,
    progress=False
)

if data.empty:
    raise ValueError("No stock data available.")

if isinstance(data.columns, pd.MultiIndex):
    data.columns = data.columns.get_level_values(0)

prices = data["Close"].dropna()

# Daily log returns in percentage units
returns_pct = (
    np.log(prices / prices.shift(1))
    .dropna() * 100
)

# Historical volatility
historical_vol = (
    np.sqrt(returns_pct.iloc[-20:].pow(2).mean() * 252)
)

# GARCH(1,1)
garch_model = arch_model(
    returns_pct,
    mean="Constant",
    vol="GARCH",
    p=1,
    q=1,
    dist="t"
)

garch_fit = garch_model.fit(disp="off")

garch_forecast = garch_fit.forecast(
    horizon=HORIZON
)

garch_variance = (
    garch_forecast.variance.iloc[-1].to_numpy()
)

garch_mean = (
    garch_forecast.mean.iloc[-1].to_numpy()
)

garch_vol = np.sqrt(
    np.mean(garch_variance + garch_mean**2) * 252
)

# Build XGBoost features
features = pd.DataFrame(index=returns_pct.index)

for window in [5, 10, 20, 60]:
    features[f"var_{window}d"] = (
        returns_pct.pow(2).rolling(window).mean()
    )

    features[f"return_{window}d"] = (
        returns_pct.rolling(window).sum()
    )

features["abs_return_1d"] = returns_pct.abs()
features["return_1d"] = returns_pct
features["negative_return_1d"] = (
    returns_pct < 0
).astype(int)

# Future five-day realized variance target
future = pd.concat(
    [
        returns_pct.shift(-i).pow(2)
        for i in range(1, HORIZON + 1)
    ],
    axis=1
)

features["target"] = future.mean(axis=1)
features.iloc[-HORIZON:, -1] = np.nan

dataset = features.dropna()

feature_cols = [
    c for c in dataset.columns if c != "target"
]

# Selected Very Simple XGBoost configuration
xgb_model = XGBRegressor(
    n_estimators=50,
    max_depth=2,
    learning_rate=0.03,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=10,
    min_child_weight=10,
    objective="reg:squarederror",
    random_state=42,
    n_jobs=2
)

xgb_model.fit(
    dataset[feature_cols],
    np.log(dataset["target"] + EPSILON)
)

# Latest features, including unlabeled dates
latest_features = (
    features[feature_cols]
    .dropna()
    .iloc[[-1]]
)

predicted_log_var = xgb_model.predict(
    latest_features
)[0]

predicted_var = max(
    np.exp(predicted_log_var) - EPSILON,
    1e-8
)

xgb_vol = np.sqrt(predicted_var * 252)

# Display results
volatility_forecasts = pd.DataFrame({
    "Method": ["Historical", "GARCH", "XGBoost"],
    "Annualized Volatility (%)": [
        historical_vol,
        garch_vol,
        xgb_vol
    ]
})

print(f"\nVOLATILITYLAB — {ticker}")
print("Five-day volatility forecasting comparison")

display(volatility_forecasts.round(2))


## 5. Illustrative five-day hedge comparison

Uses the same simulated path and initial option premium for all three inputs. This is **not** an out-of-sample historical hedging backtest.

In [ ]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Convert actual forecasts from percentages to decimals
vol_inputs = {
    "Historical": historical_vol / 100,
    "GARCH": garch_vol / 100,
    "XGBoost": xgb_vol / 100
}

# Simulate a hypothetical five-day stock path
np.random.seed(42)

S0 = 100
K = 100
T = 5 / 252
r = 0.04
days = 5
dt = T / days

# Illustrative true volatility for the simulation
true_vol = 0.35

stock_path_5d = [S0]

for i in range(days):

    shock = np.random.normal()

    new_price = stock_path_5d[-1] * np.exp(
        (r - 0.5 * true_vol**2) * dt
        + true_vol * np.sqrt(dt) * shock
    )

    stock_path_5d.append(new_price)

# Compare hedging with different volatility forecasts
results = []
hedge_paths = {}

for name, vol in vol_inputs.items():

    hedge = simulate_hedge_corrected(
        stock_path=stock_path_5d,
        K=K,
        T=T,
        r=r,
        hedge_vol=vol,
        market_vol=0.35,
        option_type="call",
        contract_size=100,
        transaction_cost=0.001
    )

    hedge_paths[name] = hedge

    results.append({
        "Model": name,
        "Forecast Volatility (%)": vol * 100,
        "Final Hedging P&L ($)": hedge["PortfolioValue"].iloc[-1],
        "Transaction Costs ($)": hedge["TransactionCosts"].iloc[-1]
    })

comparison_5d = pd.DataFrame(results)

display(comparison_5d.round(2))

# Plot hedging performance
plt.figure(figsize=(11, 5))

for name, hedge in hedge_paths.items():
    plt.plot(
        hedge["Day"],
        hedge["PortfolioValue"],
        label=name,
        linewidth=2
    )

plt.axhline(0, color="black", linestyle="--")

plt.title("Five-Day Delta Hedging — Model Comparison")
plt.xlabel("Trading Day")
plt.ylabel("Portfolio P&L ($)")
plt.legend()
plt.grid(alpha=0.3)
plt.show()


## 6. Monte Carlo comparison (500 paths)

In [ ]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Monte Carlo simulation settings
N_SIMULATIONS = 500
DAYS = 5

S0 = 100
K = 100
T = DAYS / 252
r = 0.04

# Hypothetical true market volatility
true_vol = 0.35
market_vol = 0.35

rng = np.random.default_rng(42)

simulation_results = []

for sim in range(N_SIMULATIONS):

    # Generate one hypothetical stock-price path
    shocks = rng.standard_normal(DAYS)

    stock_path_mc = [S0]

    for shock in shocks:

        next_price = stock_path_mc[-1] * np.exp(
            (r - 0.5 * true_vol**2) * (T / DAYS)
            + true_vol * np.sqrt(T / DAYS) * shock
        )

        stock_path_mc.append(next_price)

    # Hedge the same path using each volatility model
    for model_name, hedge_vol in vol_inputs.items():

        result = simulate_hedge_corrected(
            stock_path=stock_path_mc,
            K=K,
            T=T,
            r=r,
            hedge_vol=hedge_vol,
            market_vol=market_vol,
            option_type="call",
            contract_size=100,
            transaction_cost=0.001
        )

        simulation_results.append({
            "Simulation": sim,
            "Model": model_name,
            "FinalPnL": result["PortfolioValue"].iloc[-1],
            "TransactionCosts": result["TransactionCosts"].iloc[-1]
        })

mc_df = pd.DataFrame(simulation_results)

# Summarize results
mc_summary = mc_df.groupby("Model").agg(
    MeanPnL=("FinalPnL", "mean"),
    StdPnL=("FinalPnL", "std"),
    MeanAbsolutePnL=("FinalPnL", lambda x: np.mean(np.abs(x))),
    MeanCosts=("TransactionCosts", "mean"),
    WorstPnL=("FinalPnL", "min")
).reset_index()

display(mc_summary.round(2))

# Visualize hedging P&L distributions
plt.figure(figsize=(11, 5))

for model_name in vol_inputs:

    values = mc_df[
        mc_df["Model"] == model_name
    ]["FinalPnL"]

    plt.hist(
        values,
        bins=35,
        alpha=0.45,
        label=model_name
    )

plt.title("Monte Carlo — Hedging P&L Distribution")
plt.xlabel("Final Hedging P&L ($)")
plt.ylabel("Frequency")
plt.legend()
plt.grid(alpha=0.3)
plt.show()


## 7. Volatility-regime sensitivity analysis

In [ ]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Volatility scenarios
scenarios = {
    "Low": 0.20,
    "Moderate": 0.35,
    "High": 0.60
}

N_SIMULATIONS = 500
DAYS = 5

S0 = 100
K = 100
T = DAYS / 252
r = 0.04

rng = np.random.default_rng(42)

# Use the actual model forecasts from earlier
vol_inputs = {
    "Historical": historical_vol / 100,
    "GARCH": garch_vol / 100,
    "XGBoost": xgb_vol / 100
}

# Common random shocks for fair comparisons
shocks = rng.standard_normal(
    (N_SIMULATIONS, DAYS)
)

stress_results = []

for scenario_name, true_vol in scenarios.items():

    print(f"Running {scenario_name} volatility scenario...")

    for sim in range(N_SIMULATIONS):

        # Simulate stock prices
        stock_path = [S0]

        for shock in shocks[sim]:

            next_price = stock_path[-1] * np.exp(
                (r - 0.5 * true_vol**2) * (T / DAYS)
                + true_vol * np.sqrt(T / DAYS) * shock
            )

            stock_path.append(next_price)

        # Compare hedging strategies
        for model_name, hedge_vol in vol_inputs.items():

            hedge = simulate_hedge_corrected(
                stock_path=stock_path,
                K=K,
                T=T,
                r=r,
                hedge_vol=hedge_vol,
                market_vol=true_vol,
                option_type="call",
                contract_size=100,
                transaction_cost=0.001
            )

            stress_results.append({
                "Scenario": scenario_name,
                "Simulation": sim,
                "Model": model_name,
                "FinalPnL": hedge["PortfolioValue"].iloc[-1],
                "Costs": hedge["TransactionCosts"].iloc[-1]
            })

stress_df = pd.DataFrame(stress_results)

# Summarize hedging performance
stress_summary = stress_df.groupby(
    ["Scenario", "Model"]
).agg(
    MeanPnL=("FinalPnL", "mean"),
    MeanAbsolutePnL=(
        "FinalPnL",
        lambda x: np.mean(np.abs(x))
    ),
    StdPnL=("FinalPnL", "std"),
    MeanCosts=("Costs", "mean")
).reset_index()

display(stress_summary.round(2))


## 8. Regime comparison chart

In [ ]:

import matplotlib.pyplot as plt

# Organize results for comparison
pivot = stress_summary.pivot(
    index="Scenario",
    columns="Model",
    values="MeanAbsolutePnL"
)

# Arrange scenarios
pivot = pivot.reindex([
    "Low",
    "Moderate",
    "High"
])

# Create comparison graph
pivot.plot(
    kind="bar",
    figsize=(10, 5)
)

plt.title("Hedging Error Across Volatility Scenarios")
plt.xlabel("Market Volatility Scenario")
plt.ylabel("Mean Absolute Hedging P&L ($)")
plt.xticks(rotation=0)
plt.legend(title="Hedging Method")
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()


## 9. Interactive hedging demonstration

Change option type, model, strike, and trading costs. Uses the fixed five-day simulated path and fixed volatility estimates from Section 4.

In [ ]:

import ipywidgets as widgets
from IPython.display import display, clear_output
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Use forecasts from Feature 1
vol_inputs = {
    "Historical": historical_vol / 100,
    "GARCH": garch_vol / 100,
    "XGBoost": xgb_vol / 100
}

# Interactive controls
option_selector = widgets.Dropdown(
    options=["call", "put"],
    value="call",
    description="Option:"
)

model_selector = widgets.Dropdown(
    options=list(vol_inputs.keys()),
    value="GARCH",
    description="Model:"
)

strike_slider = widgets.IntSlider(
    value=100,
    min=70,
    max=130,
    step=5,
    description="Strike:"
)

cost_slider = widgets.FloatSlider(
    value=0.10,
    min=0,
    max=0.50,
    step=0.05,
    description="Cost (%):"
)

run_button = widgets.Button(
    description="Run Hedging Simulation",
    button_style="success"
)

output = widgets.Output()

def run_interactive_simulation(button):

    with output:
        clear_output(wait=True)

        selected_model = model_selector.value
        selected_vol = vol_inputs[selected_model]

        # Reuse the five-day simulated stock path
        result = simulate_hedge_corrected(
            stock_path=stock_path_5d,
            K=strike_slider.value,
            T=5/252,
            r=0.04,
            hedge_vol=selected_vol,
            market_vol=0.35,
            option_type=option_selector.value,
            contract_size=100,
            transaction_cost=cost_slider.value / 100
        )

        final_pnl = result["PortfolioValue"].iloc[-1]
        total_cost = result["TransactionCosts"].iloc[-1]

        print("VOLATILITYLAB — INTERACTIVE HEDGING")
        print("-----------------------------------")
        print("Model:", selected_model)
        print("Option:", option_selector.value)
        print("Strike:", strike_slider.value)
        print(f"Hedge volatility: {selected_vol*100:.2f}%")
        print(f"Final P&L: ${final_pnl:.2f}")
        print(f"Transaction costs: ${total_cost:.2f}")

        plt.figure(figsize=(10, 5))
        plt.plot(
            result["Day"],
            result["PortfolioValue"],
            linewidth=2
        )

        plt.axhline(
            0,
            color="black",
            linestyle="--"
        )

        plt.title(
            f"{selected_model} — "
            f"{option_selector.value.title()} Hedge"
        )
        plt.xlabel("Trading Day")
        plt.ylabel("Portfolio P&L ($)")
        plt.grid(alpha=0.3)
        plt.show()

run_button.on_click(run_interactive_simulation)

display(
    widgets.VBox([
        widgets.HTML("<h3>VolatilityLab Hedging Simulator</h3>"),
        option_selector,
        model_selector,
        strike_slider,
        cost_slider,
        run_button,
        output
    ])
)


## Interpretation and limitations

- **Hedging objective:** Compare the distribution of final delta-hedged portfolio P&L and trading costs under alternative volatility inputs. Lower mean absolute P&L indicates smaller deviations, not necessarily higher profits.
- **Forecast horizons:** The model estimates and hedge simulation use a five-trading-day horizon.
- **Experimental limitations:** Stock paths are simulated using constant scenario volatility; the volatility forecasts remain fixed during each path. Market option premiums are modeled using Black–Scholes rather than observed option quotes. GARCH/XGBoost differences here are sensitivity results, not causal evidence that one model hedges better in live markets.
- **Model assumptions:** European-style options, no dividends, fixed risk-free rate, daily rebalancing, fractional shares, and simplified proportional transaction costs.
 